In [33]:
!pip install -q google-api-python-client

In [34]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import re

In [35]:
auth.authenticate_user()

In [36]:
MARKDOWN_CONTENT = """# Product Team Sync - May 15, 2023

## Attendees

- Sarah Chen (Product Lead)
- Mike Johnson (Engineering)
- Anna Smith (Design)
- David Park (QA)

## Agenda

### 1. Sprint Review

* Completed Features
  * User authentication flow
  * Dashboard redesign
  * Performance optimization
    * Reduced load time by 40%
    * Implemented caching solution
* Pending Items
  * Mobile responsive fixes
  * Beta testing feedback integration

### 2. Current Challenges

* Resource constraints in QA team
* Third-party API integration delays
* User feedback on new UI
  * Navigation confusion
  * Color contrast issues

### 3. Next Sprint Planning

* Priority Features
  * Payment gateway integration
  * User profile enhancement
  * Analytics dashboard
* Technical Debt
  * Code refactoring
  * Documentation updates

## Action Items

- [ ] @sarah: Finalize Q3 roadmap by Friday
- [ ] @mike: Schedule technical review for payment integration
- [ ] @anna: Share updated design system documentation
- [ ] @david: Prepare QA resource allocation proposal

## Next Steps

* Schedule individual team reviews
* Update sprint board
* Share meeting summary with stakeholders

## Notes

* Next sync scheduled for May 22, 2023
* Platform demo for stakeholders on May 25
* Remember to update JIRA tickets

---

Meeting recorded by: Sarah Chen
Duration: 45 minutes"""

In [55]:
def parse_markdown(content):
    index = 1
    lines = content.split('\n')
    requests = []

    for line in lines:
        line_content = line.rstrip()

        if not line_content:
            requests.append({'insertText': {'location': {'index': index}, 'text': '\n'}})
            index += 1
            continue

        # Headers
        if line_content.startswith('# ') and not line_content.startswith('## '):
            text = line_content[2:] + '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            requests.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text) - 1},
                    'paragraphStyle': {'namedStyleType': 'HEADING_1'},
                    'fields': 'namedStyleType'
                }
            })
            index += len(text)

        elif line_content.startswith('## '):
            text = line_content[3:] + '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            requests.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text) - 1},
                    'paragraphStyle': {'namedStyleType': 'HEADING_2'},
                    'fields': 'namedStyleType'
                }
            })
            index += len(text)

        elif line_content.startswith('### '):
            text = line_content[4:] + '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            requests.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text) - 1},
                    'paragraphStyle': {'namedStyleType': 'HEADING_3'},
                    'fields': 'namedStyleType'
                }
            })
            index += len(text)

        # Horizontal Rule
        elif line_content.startswith('---'):
            text = '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            index += len(text)

        # Detect lines starting with bullets or indentation
        elif line_content.lstrip().startswith(('- ', '* ', '- [ ]')):
            stripped_line = line_content.lstrip()

            # Indentation Calculation
            if line_content.startswith('\t'):
                indent_level = len(line_content) - len(stripped_line)
            else:
                spaces = len(line_content) - len(stripped_line)
                indent_level = (spaces + 1) // 2

            is_checkbox = stripped_line.startswith('- [ ]')
            if is_checkbox:
                marker_len = 6 if stripped_line.startswith('- [ ] ') else 5
                content_text = stripped_line[marker_len:]
                preset = 'BULLET_CHECKBOX'
            else:
                content_text = stripped_line[2:]
                preset = 'BULLET_DISC_CIRCLE_SQUARE'

            text_to_insert = ('\t' * indent_level) + content_text + '\n'

            start_idx = index
            requests.append({
                'insertText': {
                    'location': {'index': index},
                    'text': text_to_insert
                }
            })

            # Create Bullet
            requests.append({
                'createParagraphBullets': {
                    'range': {'startIndex': start_idx, 'endIndex': start_idx + len(text_to_insert) - 1},
                    'bulletPreset': preset
                }
            })

            # Handle @Mentions
            if '@' in content_text:
                for match in re.finditer(r'@(\w+)', content_text):
                    mention_start = start_idx + match.start()
                    mention_end = start_idx + match.end()

                    requests.append({
                        'updateTextStyle': {
                            'range': {'startIndex': mention_start, 'endIndex': mention_end},
                            'textStyle': {'bold': True, 'foregroundColor': {'color': {'rgbColor': {'red': 0.2, 'green': 0.4, 'blue': 0.8}}}},
                            'fields': 'bold,foregroundColor'
                        }
                    })

            index += len(content_text) + 1

        # Footer Info 
        elif any(line_content.startswith(prefix) for prefix in ['Meeting recorded by:', 'Duration:']):
            text = line_content + '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            requests.append({
                'updateTextStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text) - 1},
                    'textStyle': {'italic': True, 'foregroundColor': {'color': {'rgbColor': {'red': 0.5, 'green': 0.5, 'blue': 0.5}}}},
                    'fields': 'italic,foregroundColor'
                }
            })
            index += len(text)

        # Default Text 
        else:
            text = line_content + '\n'
            requests.append({'insertText': {'location': {'index': index}, 'text': text}})
            index += len(text)

    return requests

In [56]:
def create_google_doc(title, content):
    """
    Creates a new Google Doc with formatted content

    Args:
        title: Document title
        content: Markdown content to convert

    Returns:
        Document ID of created doc
    """
    try:
        # Build the Docs API service
        service = build('docs', 'v1')

        # Create a new document
        doc = service.documents().create(body={'title': title}).execute()
        doc_id = doc.get('documentId')
        print(f'Created document: https://docs.google.com/document/d/{doc_id}/edit')

        # Parse and format the content
        requests = parse_markdown(content)

        # Apply formatting
        if requests:
            service.documents().batchUpdate(
                documentId=doc_id,
                body={'requests': requests}
            ).execute()
            print('Document formatted successfully!')

        return doc_id

    except HttpError as error:
        print(f'An error occurred: {error}')
        return None

In [57]:
if __name__ == '__main__':
    print("Starting markdown to Google Docs conversion...")
    doc_id = create_google_doc('Product Team Sync - May 15, 2023', MARKDOWN_CONTENT)

    if doc_id:
        print(f"\n✓ Success! Open your document here:")
        print(f"https://docs.google.com/document/d/{doc_id}/edit")
    else:
        print("\n✗ Failed to create document. Please check the error messages above.")

Starting markdown to Google Docs conversion...


Created document: https://docs.google.com/document/d/1KxoZYjVURE3m-d6GWHNPc1Deao4JHc_7ZupnfGzDStA/edit
Document formatted successfully!

✓ Success! Open your document here:
https://docs.google.com/document/d/1KxoZYjVURE3m-d6GWHNPc1Deao4JHc_7ZupnfGzDStA/edit
